# Aviso de abandono: tu primer modelo de machine learning

Este cuaderno aprende a adivinar **qué alumno está por dejar la academia**, mirando
cómo venía las cuatro semanas anteriores. Si funciona, el profe puede llamarlo a
tiempo, en lugar de enterarse cuando ya se fue.

Está escrito para aprender: cada paso dice qué hace y por qué. Es lo mismo que
vas a ver en los primeros módulos del técnico (pandas y scikit-learn).

**Qué quiere decir «abandonar» aquí:** que el alumno no hizo nada en la
plataforma durante las **tres semanas siguientes**: ni clase, ni ejercicios, ni
abrió una página. La consulta que arma los datos ya saca los casos que no
cuentan (las cuentas temporales y los grupos cuyo profe dejó de dar clase).

**Antes de empezar:**
1. En el editor SQL de Supabase corre `herramientas/ia/abandono-exportar.sql`.
2. Baja el resultado con «Download CSV» y guárdalo como
   `herramientas/ia/datos/abandono.csv` (esa carpeta no entra a git).
3. Si usas Google Colab, la celda de carga te pide el archivo.

Los datos salen **anónimos** (sin nombre, correo ni id), pero siguen siendo de
alumnos, muchos menores de edad: no los compartas ni los subas a ningún lado.

## 1. Cargar los datos

In [ ]:
import os
import numpy as np
import pandas as pd

ARCHIVO = "datos/abandono.csv"

if not os.path.exists(ARCHIVO):
    try:                                   # en Google Colab: se sube a mano
        from google.colab import files
        subido = files.upload()
        ARCHIVO = next(iter(subido))
    except ImportError:
        raise FileNotFoundError(
            "Falta datos/abandono.csv: córrelo con la consulta abandono-exportar.sql")

datos = pd.read_csv(ARCHIVO)
datos.head()

Cada fila es **un alumno en un lunes** (el «corte»). Las columnas cuentan cómo
venía en las cuatro semanas antes de ese lunes; la última, `se_callo`, dice si
después desapareció tres semanas (1) o no (0).

| Columna | Qué es |
|---|---|
| `alumno` | un número sorteado en cada exportación (no dice quién es) |
| `corte` | la semana, contando desde la primera |
| `dias_activos_28`, `dias_activos_7` | días con algo de actividad en las últimas 4 semanas / la última |
| `dias_sin_venir` | días desde la última vez que hizo algo |
| `dias_desde_la_primera` | cuánto lleva en la plataforma |
| `ejercicios_28`, `ejercicios_7` | ejercicios de Entrenamiento |
| `clases_28` | clases en vivo a las que entró |
| `minutos_tarde_prom` | cuánto llega tarde a clase, en promedio |
| `tareas_28` | tareas que le mandaron |
| `cobros_atrasados` | cobros vencidos sin pagar del todo |
| `elo`, `con_profe` | su Elo (si lo tiene) y si tiene profe |
| `se_callo` | **lo que queremos adivinar** |

In [ ]:
print(f"{len(datos)} filas de {datos.alumno.nunique()} alumnos")
print(f"Se callaron: {datos.se_callo.sum()} ({datos.se_callo.mean():.0%})")

Fíjate en el porcentaje: si alrededor del 12 % se calla, un «modelo» que diga
siempre «nadie se va» acierta el 88 % de las veces… y no sirve para nada. Por
eso aquí no se mide cuántas veces acierta, sino **qué tan bien ordena** a los
alumnos de más riesgo a menos (paso 4).

## 2. Preparar los datos

Los modelos no aceptan casillas vacías. El Elo falta en muchos alumnos, y que
falte también dice algo; por eso se agrega una columna `tiene_elo` y el vacío se
llena con la mediana. A quien nunca había venido antes del corte se le ponen 28
días sin venir.

In [ ]:
datos["tiene_elo"] = datos.elo.notna().astype(int)
datos["elo"] = datos.elo.fillna(datos.elo.median())
datos["dias_sin_venir"] = datos.dias_sin_venir.fillna(28)

X = datos.drop(columns=["alumno", "corte", "se_callo"])   # lo que el modelo mira
y = datos.se_callo                                       # lo que tiene que adivinar
grupos = datos.alumno
X.describe().round(1)

## 3. Separar para entrenar y para probar, por alumno

Un modelo se prueba con datos que **no vio** al aprender. Pero ojo: el mismo
alumno aparece varias semanas. Si una semana suya cae al entrenar y otra al
probar, el modelo «reconoce» al alumno y parece mejor de lo que es: es como
estudiar con las respuestas del examen.

Por eso se separa **por alumno** (`GroupKFold`): se parten los alumnos en 5
grupos, se entrena con 4 y se prueba con el que queda, cinco veces. Así cada
alumno se prueba una vez, siempre con un modelo que nunca lo vio.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict

cv = GroupKFold(n_splits=5)

def probar(modelo):
    """Devuelve, para cada fila, la probabilidad de que se calle según un
    modelo que no vio a ese alumno al entrenar."""
    return cross_val_predict(modelo, X, y, groups=grupos, cv=cv,
                             method="predict_proba")[:, 1]

## 4. Cómo se mide

Dos números:

- **AUC** (de 0,5 a 1): si tomas un alumno que se fue y otro que no, cuántas
  veces el modelo le da más riesgo al que se fue. 0,5 es tirar una moneda; 1 es
  perfecto.
- **Acierto en el 10 % más alto**: de los alumnos que el modelo marca con más
  riesgo, cuántos se fueron de verdad. Es lo que le importa al profe: si le
  avisas de 10, ¿cuántas llamadas valieron la pena?

In [ ]:
from sklearn.metrics import roc_auc_score

def medir(nombre, riesgo):
    riesgo = np.asarray(riesgo, dtype=float)
    top = np.argsort(-riesgo)[: len(riesgo) // 10]
    print(f"{nombre:30s} AUC {roc_auc_score(y, riesgo):.2f}   "
          f"acierto en el 10 % más alto: {y.iloc[top].mean():.0%}")

## 5. Primero, una regla simple

Antes de cualquier modelo, una regla que se le ocurre a cualquiera: **«el que
lleva más días sin venir es el que se va»**. Si el modelo no le gana a esto, no
vale la pena. Esta comparación es lo que más se olvida y lo que más se nota en
una entrevista.

In [ ]:
medir("Regla: días sin venir", datos.dias_sin_venir)

## 6. Tres modelos

- **Regresión logística:** el más simple. Le pone un peso a cada columna y los
  suma. Fácil de explicar. Necesita las columnas en la misma escala
  (`StandardScaler`).
- **Bosque aleatorio:** cientos de árboles de decisión que votan. Encuentra
  combinaciones («pocas clases *y* muchos días sin venir») que la logística no ve.
- **Gradient boosting:** árboles que se corrigen uno al otro. Suele ser el más
  fuerte con tablas, pero con pocos datos se pasa de listo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelos = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Bosque aleatorio": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, random_state=0),
}

medir("Regla: días sin venir", datos.dias_sin_venir)
riesgos = {}
for nombre, modelo in modelos.items():
    riesgos[nombre] = probar(modelo)
    medir(nombre, riesgos[nombre])

**Cómo leerlo:** con los datos de octubre de 2026, la regla ya daba un AUC de
0,84 y los modelos llegaban a 0,88: mejoran, pero poco. Es normal con siete
semanas de datos. Cada mes que pase, vuelve a exportar y a correr esto: con más
alumnos que se fueron, el modelo aprende más y la distancia con la regla debería
crecer. Si no crece, la regla simple alcanza, y también eso es un resultado.

## 7. ¿Qué mira el modelo?

Un modelo que nadie entiende no se usa. La regresión logística dice cuánto pesa
cada columna: positivo sube el riesgo, negativo lo baja (todas en la misma
escala, así que se pueden comparar).

In [ ]:
logistica = modelos["Regresión logística"].fit(X, y)
pesos = pd.Series(logistica[-1].coef_[0], index=X.columns).sort_values()
pesos.round(2)

In [ ]:
import matplotlib.pyplot as plt

ax = pesos.plot.barh(figsize=(7, 5), color=["#2f6f4f" if p < 0 else "#b3412b" for p in pesos])
ax.set_xlabel("Peso en la regresión logística (escala común)")
ax.set_title("Qué sube (derecha) y qué baja (izquierda) el riesgo de abandonar")
plt.tight_layout()

## 8. Elegir cuándo avisar

El modelo da un número de 0 a 1. Hay que decidir desde qué número se avisa. Si
el umbral es bajo, avisa de muchos (pocos se escapan, pero el profe recibe
muchas falsas alarmas); si es alto, avisa de pocos y casi siempre con razón,
pero se escapan más. La tabla muestra el trato para el bosque aleatorio:

In [ ]:
riesgo = riesgos["Bosque aleatorio"]
filas = []
for umbral in [0.2, 0.3, 0.4, 0.5, 0.6]:
    avisa = riesgo >= umbral
    filas.append({
        "umbral": umbral,
        "avisos": int(avisa.sum()),
        "de esos, se fueron": f"{y[avisa].mean():.0%}" if avisa.any() else "-",
        "de los que se fueron, avisó a": f"{avisa[y == 1].mean():.0%}",
    })
pd.DataFrame(filas)

## 9. Lo que falta para usarlo de verdad

Este cuaderno **mide** si se puede adivinar. Para que el profe vea el aviso en
su panel falta:

1. Una función en la base que calcule las mismas columnas **para hoy**, sin la
   respuesta (todavía no se sabe si se va a callar).
2. Pasar los pesos del modelo elegido a esa función, o empezar con la regla
   simple, que no necesita nada de esto.
3. Mostrarle al profe **por qué** avisa («lleva 12 días sin venir y dejó los
   ejercicios»), no solo un número.
4. Decirlo en la política de privacidad: la Ley 8968 pide contar para qué se usan
   los datos, y aquí se usan para algo nuevo.

**Con cuidado:** el aviso es para que alguien llame y pregunte cómo está el
alumno, nunca para cobrarle, sacarlo de un grupo ni tratarlo distinto.

## Para practicar

- Agrega una columna nueva a la consulta (por ejemplo, si su racha de días se
  cortó) y mira si sube el AUC.
- Prueba quitar `dias_sin_venir`: ¿cuánto pierde el modelo? Eso dice cuánto
  depende de una sola columna.
- Cambia «tres semanas» por dos o por cuatro en la consulta: ¿se vuelve más
  fácil o más difícil de adivinar?